## Phase 1: Browser Initialization & Network Handshake
This phase covers launching your automation instance and establishing a secure connection with the public government server.

| Encountered Error | Real-World Cause | Production-Grade Mitigation Strategy |
| :--- | :--- | :--- |
| `SessionNotCreatedException` | Local Chrome browser auto-updated in the background, breaking driver version compatibility. | Use `webdriver-manager` to auto-fetch matching binaries, or switch to Playwright which auto-manages browser binaries. |
| `WebDriverException` | Local system path conflicts, missing binary execution permissions, or lack of system memory. | Wrap initialization in a `try-except` block; catch the error, log system stats, and gracefully exit. |
| `HTTP 403 Forbidden / CAPTCHA` | The Web Application Firewall (WAF) caught standard Selenium automation headers and blocked your IP. | Deploy `undetected-chromedriver` or use Playwright with stealth plugins to wipe out automation markers. |
| `MaxRetryError / Name Resolution` | Your local machine loses internet connection, or your local DNS fails to resolve the server host. | Implement an exponential backoff retry loop utilizing the `tenacity` library before allowing the main script to fail. |

---

## Phase 2: Server Verification & Target Navigation
This phase covers validating that the government site is actively functional and navigating through categories without breaking execution.

| Encountered Error | Real-World Cause | Production-Grade Mitigation Strategy |
| :--- | :--- | :--- |
| `HTTP 502 / 503 / 504` | Government server is overloaded, crashed, or down for scheduled nightly system maintenance. | Read `driver.title` or check page text for keywords like "Maintenance"; log an alert and exit safely. |
| `HTTP 429 Too Many Requests` | Your bot is navigating too fast, tripping the server's public rate-limiting traffic rules. | Avoid hardcoded loops. Introduce random download delays (jitter) between actions to mimic natural human behavior. |
| `NoSuchElementException` | The specific category link, button, or search field identifier has changed due to an unannounced website layout update. | Use explicit conditional checks (e.g., `if driver.find_elements(...)`) rather than blindly trying to click elements. |
| `TimeoutException` | The search portal takes too long to populate query results due to slow, unoptimized legacy backend databases. | Replace `time.sleep()` with Explicit Waits (`WebDriverWait` with `expected_conditions`) to await elements dynamically. |

---

## Phase 3: Live Data Extraction (Scraping)
This phase covers reading the populated text records off the target pages while keeping the script completely fault-tolerant.

| Encountered Error | Real-World Cause | Production-Grade Mitigation Strategy |
| :--- | :--- | :--- |
| `StaleElementReferenceException` | The site dynamically updates components via AJAX in the background while your script reads the live DOM tree. | Do not parse live elements. Pull the raw page via `driver.page_source`, hand it to Selectolax, and parse it completely offline. |
| `AttributeError / NoneType` | Public data records are structurally inconsistent; a critical field (like an expiration date) is entirely blank on some rows. | Use defensive parsing tricks like `.get_text(strip=True)` inside granular `try-except` wrappers to map missing values to "N/A". |
| `IndexError` | Expected table columns or layout blocks are missing completely because a specific record uses an alternative template layout. | Check sequence boundaries using length validations (`if len(data_cells) >= expected_count:`) before slicing arrays. |

---

## Phase 4: Data Engineering & File Serialization
This phase covers structuring your scraped fields safely and committing them permanently to your disk storage layers.

| Encountered Error | Real-World Cause | Production-Grade Mitigation Strategy |
| :--- | :--- | :--- |
| `PermissionError / File Locked` | Your script tries to write data out to your destination file, but you accidentally left that target CSV open in Microsoft Excel. | Write data records out into uniquely timestamped files, or catch the exception and prompt the user to free the disk lock. |
| `UnicodeEncodeError` | Government data contains specialized localized characters or symbols that cannot map to a standard ASCII file encoder. | Explicitly enforce universal compatibility by initializing file streams with the `encoding="utf-8"` parameter flag. |
| `Total Data Loss on Script Crash` | The script hits a formatting error on record #900 and crashes out, wiping all earlier data held in volatile RAM cache memory. | Open files using Append Mode (`"a"`). Write each processed data row onto the disk storage array immediately inside the loop. |

In [ ]:
from selenium import webdriver # browser automation
from selenium.webdriver.chrome.options import Options # adding preferences to chrome driver (i.e block images etc)
from selenium.webdriver.common.by import By # to locate element from the webpages 
from selenium.webdriver.support.ui import WebDriverWait # for handling slow websites 
from selenium.webdriver.support import expected_conditions as EC # handling crashes (locating the element presence and clicking element)
from selenium.common.exceptions import WebDriverException # error class
import pandas as pd 
import os # file handling
import re # regex (for locating specific keywords)
import time # for limiting the request speed 
import csv # for saving the data in csv format
from bs4 import BeautifulSoup # for parsing the html data


def create_driver():
    """Spawns a fresh, optimized Chrome WebDriver session."""
    options = Options()
    # run in headless mode (background) without opening a physical browser window
    options.add_argument("--headless=new")
    # save memory and cpu usage
    options.add_argument("--disable-gpu")
    # bypass os security
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    # speed up page load times by blocking all image content from downloading
    options.add_experimental_option("prefs", {"profile.managed_default_content_settings.images": 2})
    return webdriver.Chrome(options=options)

# fucntion for navigating to district zones and locating to FPS 
def navigate_to_district(driver, url, zone_name):
    """
    Executes the full 4-step sequence to reach a district's FPS list page cleanly:
    URL -> Click GOA -> Click District (e.g. SOUTH GOA) -> Click FAIR PRICE SHOPS
    """
    driver.get(url)

    # 1 click GOA on main map
    goa_element = WebDriverWait(driver, 15).until(
        EC.presence_of_element_located((By.XPATH, "//a[contains(@title, 'GOA')] | //img[contains(@aria-label, 'GOA')]"))
    )
    driver.execute_script("arguments[0].click();", goa_element)

    # verify are we on the right page or not 
    WebDriverWait(driver, 10).until(
        EC.text_to_be_present_in_element((By.CSS_SELECTOR, ".status.m_menu"), "GOA")
    )

    # 2 click specific District Zone (NORTH GOA or SOUTH GOA)
    zone_element = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.XPATH, f"//a[@title='{zone_name}'] | //a[@aria-label='{zone_name}']"))
    )
    onclick_script = zone_element.get_attribute("onclick")
    driver.execute_script(onclick_script if onclick_script else "arguments[0].click();", zone_element)

    # 3 click FAIR PRICE SHOPS link after locating to specific zone
    fair_price_shop = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.XPATH, "//a[contains(@onclick, 'liveFpsdata')]"))
    )
    driver.execute_script(fair_price_shop.get_attribute("onclick"))

    # 4 extract all FPS onclick actions
    fps_list_container = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, "ul.menu")) # get all the fps codes 
    )
    fps_elements = fps_list_container.find_elements(By.CSS_SELECTOR, "li.menu_list a")
    fps_actions = [elem.get_attribute("onclick") for elem in fps_elements if elem.get_attribute("onclick")]
    
    return fps_actions

# -- MAIN  --
years = [2026] #  Specific year
months = [3, 4] # month (march and april)
goa_zones = ['NORTH GOA', 'SOUTH GOA'] # zones in goa 

driver = create_driver()

for year in years: # loop over year (2026) 
    for month in months: #loop over month (march and april)

        # website link (dynamically changes month and year as per search needs)
        url = f"https://impds.nic.in/sale/stateUnautmated?month={month}&year={year}#"

        # loop over north zone and then follwed by south zone
        for zone_name in goa_zones:
            formatted_zone = zone_name.replace(" ", "_") # chnge the name from NORTH GOA TO NORTH_GOA (for folder)
            """
            Create a new folder
            raw_html /
            |---year_month (2026_3)/
            |-------------------------/NORTH_GOA
            |------------------------------------/....FPS RAW DATA
            """
            folder_path = f"raw_html/{year}_{month}/{formatted_zone}" 
            os.makedirs(folder_path, exist_ok=True)

            # Step 1 NAVIGATE TO DISTRICT PAGE 
            fps_actions = [] # create a list to store all the fps code 
            while not fps_actions: # if not in fps code , get the data 
                try:
                    print(f"\nNavigating to {zone_name} ({month}/{year})...")
                    fps_actions = navigate_to_district(driver, url, zone_name) # function call 
                # If Chrome session runs out or faces any error     
                except (WebDriverException, Exception) as nav_err: 
                    print(f"Page navigation failed for {zone_name}. Rebooting Chrome... ({nav_err})")
                    try:
                        driver.quit() # quit the chrome driver 
                    except Exception:
                        pass
                    driver = create_driver() # again restart the chrome drive 

            total_fps = len(fps_actions) #total fps
            print(f"Extracting {total_fps} items for {zone_name} ({month}/{year})...")
            start_time = time.time()

            # -- STEP 2 LOOP THROUGH ALL FPS ITEMS WITH AUTO RECOVERY --

            """
            this script is loops through a list of text actions, extracts a unique identification number (id)
            from each action using a regular expression (regex), and sets up a unique HTML file path to save data for each item
            fps_action - list containing all the fps code

            """
            # index holds the current iteration count (1-based), action holds the onclick script for the current FPS item
            for index, action in enumerate(fps_actions, start=1): # loop setup tracking each iteam count fom 1
                # search for a numeric ID within the action string using regex
                fps_id_match = re.search(r"'\s*(\d+)\s*'", action)
                # if a match is found, extract the ID; otherwise, use a fallback name based on the index
                fps_id = fps_id_match.group(1) if fps_id_match else f"item_{index}" 
                target_file = f"{folder_path}/fps_{fps_id}.html" # store the data in the folder path with the fps_<fps_id>.html

                # Skip downloaded files
                if os.path.exists(target_file):
                    continue

                success = False # flag to indicate if the current item was downloaded successfully
                attempts = 0 # counter to track the number of attempts made to download the current item

                """
                The following loop that will keep running as long as success is still False and the script has tried fewer than 3 times attemps to fetch data for the current FPS item. This is a retry mechanism to handle potential issues like network errors, timeouts, or unexpected page behavior. If failed it will restart the entire Chrome session and re-navigate back to the district page to try again. This ensures that even if a single FPS item fails to load, the script can recover and continue processing the remaining items without manual intervention.
                """
                while not success and attempts < 3:
                    # Get the FPS data by executing the JavaScript action and saving the resulting HTML to a file
                    try:
                        driver.execute_script(action)
                        time.sleep(0.1)

                        # Grab innerHTML of target data panel
                        """
                        The following block of code attempts to locate a specific data panel on the webpage using a CSS selector. If the panel is found, it retrieves its inner HTML content. If the panel is not found or an error occurs, it falls back to capturing the entire page source. The retrieved HTML content is then saved to a target file for later analysis or processing.
                        """
                        try:
                            data_panel = driver.find_element(By.CSS_SELECTOR, ".col-md-9, .col-md-8, #fpsData, .padding-top-10")
                            html_content = data_panel.get_attribute("innerHTML")
                        except Exception:
                            html_content = driver.page_source

                        with open(target_file, "w", encoding="utf-8") as f:
                            f.write(html_content)

                        success = True # once data is fetch move to next 


                    # Rebot entire session if the attempts exceeds more than 3 times (network error, timeout, or unexpected page behavior)
                    except (WebDriverException, Exception) as err:
                        attempts += 1
                        print(f"Session dropped at {zone_name} item {index}/{total_fps}. Rebooting (Attempt {attempts})...")
                        try:
                            driver.quit() # quit the chrome driver
                        except Exception:
                            pass
                        
                        # Full reboot & re-navigate back to the district
                        driver = create_driver()
                        try:
                            fps_actions = navigate_to_district(driver, url, zone_name)
                        except Exception as restore_err:
                            print(f"Failed to restore district position: {restore_err}")

            # Task Complete for the current zone
            print(f"Finished {zone_name} in {round(time.time() - start_time, 2)}s!")

try:
    driver.quit() # final step quit the chrome driver
except Exception:
    pass


Navigating to NORTH GOA (3/2026)...
Extracting 240 items for NORTH GOA (3/2026)...
Finished NORTH GOA in 0.01s!

Navigating to SOUTH GOA (3/2026)...
⚠️ Page navigation failed for SOUTH GOA. Rebooting Chrome... (Message: 
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff688df8945+14ce5]
	chromedriver!GetHandleVerifier [0x7ff688df89a0+14d40]
	chromedriver!(No symbol) [0x7ff688925afd]
	chromedriver!(No symbol) [0x7ff688980b59]
	chromedriver!(No symbol) [0x7ff688980e5c]
	chromedriver!(No symbol) [0x7ff6889d18d7]
	chromedriver!(No symbol) [0x7ff6889ce4ab]
	chromedriver!(No symbol) [0x7ff68897308c]
	chromedriver!(No symbol) [0x7ff688973fb3]
	chromedriver!GetHandleVerifier [0x7ff68942e60b+64a9ab]
	chromedriver!GetHandleVerifier [0x7ff689428b12+644eb2]
	chromedriver!GetHandleVerifier [0x7ff68944e3ae+66a74e]
	chromedriver!GetHandleVerifier [0x7ff688e15d7e+3211e]
	chromedriver!GetHandleVerifier [0x7ff688e1e52c+3a8cc]
	chromedriver!GetHandleVerifier [0x7ff688e02854+1ebf4]
	chromedriver!GetHandl

In [ ]:
from selenium import webdriver # browser automation
from selenium.webdriver.chrome.options import Options # adding preferences to chrome driver (i.e block images etc)
from selenium.webdriver.common.by import By # to locate element from the webpages 
from selenium.webdriver.support.ui import WebDriverWait # for handling slow websites 
from selenium.webdriver.support import expected_conditions as EC # handling crashes (locating the element presence and clicking element)
from selenium.common.exceptions import WebDriverException # error class
import pandas as pd 
import os # file handling
import re # regex (for locating specific keywords)
import time # for limiting the request speed 
import csv # for saving the data in csv format
from bs4 import BeautifulSoup # for parsing the html data
import json # for saving the data in json format


def create_driver():
    """Spawns a fresh, optimized Chrome WebDriver session."""
    options = Options()
    # run in headless mode (background) without opening a physical browser window
    options.add_argument("--headless=new")
    # save memory and cpu usage
    options.add_argument("--disable-gpu")
    # bypass os security
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    # speed up page load times by blocking all image content from downloading
    options.add_experimental_option("prefs", {"profile.managed_default_content_settings.images": 2})
    return webdriver.Chrome(options=options)

# fucntion for navigating to district zones and locating to FPS 
def navigate_to_district(driver, url, zone_name):
    """
    Executes the full 4-step sequence to reach a district's FPS list page cleanly:
    URL -> Click GOA -> Click District (e.g. SOUTH GOA) -> Click FAIR PRICE SHOPS
    """
    driver.get(url)

    # 1 click GOA on main map
    goa_element = WebDriverWait(driver, 15).until(
        EC.presence_of_element_located((By.XPATH, "//a[contains(@title, 'GOA')] | //img[contains(@aria-label, 'GOA')]"))
    )
    driver.execute_script("arguments[0].click();", goa_element)

    # verify are we on the right page or not 
    WebDriverWait(driver, 10).until(
        EC.text_to_be_present_in_element((By.CSS_SELECTOR, ".status.m_menu"), "GOA")
    )

    # 2 click specific District Zone (NORTH GOA or SOUTH GOA)
    zone_element = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.XPATH, f"//a[@title='{zone_name}'] | //a[@aria-label='{zone_name}']"))
    )
    onclick_script = zone_element.get_attribute("onclick")
    driver.execute_script(onclick_script if onclick_script else "arguments[0].click();", zone_element)

    # 3 click FAIR PRICE SHOPS link after locating to specific zone
    fair_price_shop = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.XPATH, "//a[contains(@onclick, 'liveFpsdata')]"))
    )
    driver.execute_script(fair_price_shop.get_attribute("onclick"))

    # 4 extract all FPS links
    fps_list_container = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, "ul.menu")))
    fps_elements = fps_list_container.find_elements(By.CSS_SELECTOR, "li.menu_list a") # get all the fps codes
    
    # get all the FPS links and their corresponding onclick actions and text
    fps_actions = [
        (elem.get_attribute("onclick"), elem.text.strip())
        for elem in fps_elements 
        if elem.get_attribute("onclick")
    ]
    
    return fps_actions

# -- MAIN  --
years = [2026] #  Specific year
months = [3, 4] # month (march and april)
goa_zones = ['NORTH GOA', 'SOUTH GOA'] # zones in goa 

driver = create_driver()

for year in years: # loop over year (2026) 
    for month in months: #loop over month (march and april)

        # website link (dynamically changes month and year as per search needs)
        url = f"https://impds.nic.in/sale/stateUnautmated?month={month}&year={year}#"

        # loop over north zone and then follwed by south zone
        for zone_name in goa_zones:
            formatted_zone = zone_name.replace(" ", "_") # chnge the name from NORTH GOA TO NORTH_GOA (for folder)
            """
            Create a new folder
            data /
            |---raw/
            |------year_month (2026_3)/
            |-------------------------/NORTH_GOA
            |------------------------------------/....FPS RAW DATA
            """
            folder_path = f"data/raw/{year}-{month:02d}/{formatted_zone.lower()}"
            os.makedirs(folder_path, exist_ok=True)

            # Step 1 NAVIGATE TO DISTRICT PAGE 
            fps_actions = [] # create a list to store all the fps code 
            while not fps_actions: # if not in fps code , get the data 
                try:
                    print(f"\nNavigating to {zone_name} ({month}/{year})...")
                    fps_actions = navigate_to_district(driver, url, zone_name) # function call 
                # If Chrome session runs out or faces any error     
                except (WebDriverException, Exception) as nav_err: 
                    print(f"Page navigation failed for {zone_name}. Rebooting Chrome... ({nav_err})")
                    try:
                        driver.quit() # quit the chrome driver 
                    except Exception:
                        pass
                    driver = create_driver() # again restart the chrome drive 

            total_fps = len(fps_actions) #total fps
            print(f"Extracting {total_fps} items for {zone_name} ({month}/{year})...")
            start_time = time.time()

            # -- STEP 2 LOOP THROUGH ALL FPS ITEMS WITH AUTO RECOVERY --

            """
            this script is loops through a list of text actions, extracts a unique identification number (id)
            from each action using a regular expression (regex), and sets up a unique HTML file path to save data for each item
            fps_action - list containing all the fps code

            """
            # index holds the current iteration count (1-based), action holds the onclick script for the current FPS item
            for index, (action, full_text) in enumerate(fps_actions, start=1):

                # extract the ID (158500100001)
                check_fps_id = re.search(r"^(\d+)", full_text)
                fps_id = check_fps_id.group(1)
        
                # Regex to extract the shop name 
                name_match = re.search(r":\s*([^-]+)", full_text) # match name and return the name
                raw_name = name_match.group(1).strip() if name_match else full_text

                # clean up the shop name (remove dots, slashes, punctuation, convert to lowercase underscores)
                clean_name = re.sub(r"[^\w\s]", "", raw_name)  # rmeove \ from name ( M/s -> M S)
                fps_name = "_".join(clean_name.lower().split())  # "smt_sanna_sanjay_padte" ( M S -> ms)

                # save the file 
                target_file = f"{folder_path}/{fps_id}_{fps_name}.json"

                # Skip downloaded files
                if os.path.exists(target_file):
                    continue

                # Execute your click and save logic using target_file...
                success = False # flag to indicate if the current item was downloaded successfully
                attempts = 0 # counter to track the number of attempts made to download the current item
                """
                The following block of code attempts to locate a specific data panel on the webpage using a CSS selector.
                    If the panel is found, it retrieves its inner HTML content. If the panel is not found or an error occurs, 
                    it falls back to capturing the entire page source. The retrieved HTML content is then saved to a target file for
                    later analysis or processing.
                """
                while not success and attempts < 3:
                    try:
                        # execute the javascript action to switch to the target FPS
                        driver.execute_script(action)
                        time.sleep(1.5)  # pause briefly for dynamic AJAX updates

                        # parse live page DOM with BeautifulSoup
                        soup = BeautifulSoup(driver.page_source, "html.parser")

                        # direct metadata extraction
                        cal_text = soup.find("li", id="calModal").text.strip() # get the month and year from the calModal element
                        month, year = cal_text.split("-") # separate month and year from the calModal element

                        state = ( soup.find("div", {"key": "state"}).text.replace("\n", "").strip()) # get the state name
                        district = (soup.find("div", {"key": "district"}).text.replace("\n", "").strip()) # get the district name
                        fps_id = soup.find("span", class_="counter_num4").text.strip() # get the fps id from the counter_num4 element
                        fps_name = soup.find("span", class_="counter_num3").text.strip() # get the fps name from the counter_num3 element

                        # extract summary cards
                        summary_cards = {
                            # total number of e-transactions
                            "total_etransaction": soup.select_one(".nav-block-greenlight .counter")
                            .text.strip()
                            .replace(",", ""),

                            # total number of aadhaar authenticated transactions
                            "aadhaar_authenticated": soup.select_one(".nav-block-green .counter")
                            .text.strip()
                            .replace(",", ""),

                            # total number of other mode authenticated transactions
                            "other_mode_authenticated": soup.select_one(".nav-light-pink .counter")
                            .text.strip()
                            .replace(",", ""),

                            # total number of non-authenticated transactions
                            "non_authenticated": soup.select_one(".nav-light-purple .counter")
                            .text.strip()
                            .replace(",", ""),
                        }

                        # function to parse table rows matrix
                        def parse_table_data(css_selector):
                            rows_list = []
                            table = soup.select_one(css_selector)
                            if table:
                                for tr in table.select("tbody tr, tfoot tr"):
                                    cells = tr.find_all(["td", "th"])
                                    if len(cells) >= 5:
                                        raw_label = (
                                            cells[0]
                                            .text.replace("+", "")
                                            .replace("-", "")
                                            .strip()
                                        )
                                        clean_label = re.sub(r"\s+", " ", raw_label)
                                        rows_list.append({
                                            "row_label": clean_label,
                                            "regular": cells[1].text.strip().replace(",", ""),
                                            "intra_state": cells[2]
                                            .text.strip()
                                            .replace(",", ""),
                                            "inter_state": cells[3]
                                            .text.strip()
                                            .replace(",", ""),
                                            "total": cells[4].text.strip().replace(",", ""),
                                        })
                            return rows_list

                        # fps record object to store all the data in a single json file
                        fps_record = {
                            "year": year,
                            "month": month,
                            "state": state,
                            "district": district,
                            "fps_id": fps_id,
                            "fps_name": fps_name,
                            "summary_cards": summary_cards,
                            "number_of_transactions": parse_table_data("table.state-rep0"),
                            "number_of_transacted_ration_cards": parse_table_data(
                                "table.state-rep2"
                            ),
                            "distributed_quantity_kg": parse_table_data("table.state-rep1"),
                        }

                        # save JSON file named after the specific FPS ID
                        json_filename = f"{target_file}"
                        with open(json_filename, "w", encoding="utf-8") as f:
                            json.dump(fps_record, f, indent=2, ensure_ascii=False)

                        print(
                            f"Successfully extracted & saved data for FPS ID {fps_id} to {json_filename}"
                        )
                        success = True  # Move to next FPS

                    # Session reboot error handling
                    except (WebDriverException, Exception) as err:
                        attempts += 1
                        print(
                            f"Session dropped at {zone_name} item {index}/{total_fps}. Rebooting (Attempt {attempts})... Error: {err}"
                        )
                        try:
                            driver.quit()
                        except Exception:
                            pass

                        # Full reboot & re-navigate back to district
                        driver = create_driver()
                        try:
                            navigate_to_district(driver, url, zone_name)
                        except Exception as restore_err:
                            print(f"Failed to restore district position: {restore_err}")

            # Task Complete for the current zone
            print(f"Finished {zone_name} in {round(time.time() - start_time, 2)}s!")

try:
    driver.quit() # final step quit the chrome driver
except Exception:
    pass


Navigating to NORTH GOA (3/2026)...
Extracting 240 items for NORTH GOA (3/2026)...
Successfully extracted & saved data for FPS ID 158500100001 to data/raw/2026-03/north_goa/158500100001_smtsanna_sanjay_padte.json
Successfully extracted & saved data for FPS ID 158500100002 to data/raw/2026-03/north_goa/158500100002_ms_p_s_karekar.json
Successfully extracted & saved data for FPS ID 158500100005 to data/raw/2026-03/north_goa/158500100005_arpora_nagoa_v_k_s_s_sty_ltd.json
Successfully extracted & saved data for FPS ID 158500100006 to data/raw/2026-03/north_goa/158500100006_ashok_p_shirodkar.json
Successfully extracted & saved data for FPS ID 158500100007 to data/raw/2026-03/north_goa/158500100007_ashok_g_dhargalkar.json
Successfully extracted & saved data for FPS ID 158500100008 to data/raw/2026-03/north_goa/158500100008_santosh_nagesh_prabhu.json
Successfully extracted & saved data for FPS ID 158500100009 to data/raw/2026-03/north_goa/158500100009_the_mapusa_consumer_coop_sty.json


KeyboardInterrupt: 